# AIMO 3 Submission - SC-TIR

Self-Consistency with Tool-Integrated Reasoning for AI Mathematical Olympiad.

Based on NuminaMath winning solution, adapted for AIMO 3:
- 5-digit answers (0-99999)
- H100 GPU support
- Updated kaggle_evaluation API

In [ ]:
# Install dependencies
!pip install -q vllm transformers accelerate

In [ ]:
import os
import re
import signal
import subprocess
import tempfile
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Optional

import pandas as pd
import torch
from tqdm import tqdm

IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
print(f"Is submission: {IS_SUBMISSION}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
@dataclass
class Config:
    model_id: str = "Qwen/Qwen2.5-Math-7B-Instruct"
    num_samples: int = 32        # Width: solution candidates
    num_generations: int = 4     # Depth: code execution rounds
    restart_on_fail: bool = True
    temperature: float = 0.7
    max_new_tokens: int = 2048
    timeout: int = 5             # Code execution timeout

config = Config()

In [ ]:
class PythonREPL:
    """Sandboxed Python executor with timeout."""
    
    def __init__(self, timeout: int = 5):
        self.timeout = timeout
    
    @contextmanager
    def time_limit(self, seconds):
        def handler(*_):
            raise TimeoutError(f"Timed out after {seconds}s")
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
    
    def __call__(self, code: str) -> tuple[bool, str]:
        imports = """import math
import numpy as np
import sympy as sp
from sympy import *
from fractions import Fraction
from itertools import permutations, combinations, product
"""
        code = imports + code
        
        # Ensure last line prints
        lines = code.strip().split("\n")
        if "print(" not in lines[-1]:
            lines[-1] = f"print({lines[-1].split('#')[0]})"
        code = "\n".join(lines)
        
        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "code.py")
            with open(path, "w") as f:
                f.write(code)
            
            try:
                with self.time_limit(self.timeout):
                    result = subprocess.run(
                        ["python3", path],
                        capture_output=True, text=True, timeout=self.timeout
                    )
                    if result.returncode == 0:
                        return True, result.stdout.strip()
                    return False, result.stderr.strip()[-500:]
            except Exception as e:
                return False, str(e)

executor = PythonREPL(config.timeout)

In [ ]:
def extract_boxed(text: str) -> Optional[str]:
    """Extract content from \\boxed{}."""
    idx = text.rfind("\\boxed")
    if idx < 0:
        idx = text.rfind("\\fbox")
        if idx < 0:
            return None
    
    i, depth = idx, 0
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                break
        i += 1
    else:
        return None
    
    boxed = text[idx:i+1]
    for prefix in ["\\boxed{", "\\fbox{"]:
        if boxed.startswith(prefix):
            return boxed[len(prefix):-1]
    return None


def normalize(text: str) -> str:
    """Normalize answer text."""
    if not text:
        return "-1"
    
    removals = [
        "\\text{", "}", "\\mathrm{", "$", "%", ",", " ",
        "square", "units", "degrees", "cm", "meters"
    ]
    for r in removals:
        text = text.replace(r, "")
    return text.strip()


def to_int(text: str) -> int:
    """Convert to integer in range 0-99999."""
    try:
        val = round(float(text.replace(",", "")))
        return max(0, min(99999, val))
    except:
        return -1

In [ ]:
# Load model with vLLM for fast inference
from vllm import LLM, SamplingParams

print(f"Loading model: {config.model_id}")
llm = LLM(
    model=config.model_id,
    tensor_parallel_size=torch.cuda.device_count(),
    dtype="float16",
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()
print("Model loaded!")

In [ ]:
def make_prompt(problem: str) -> str:
    """Create prompt for the model."""
    return f"""Solve this math problem step by step. Use Python code when helpful.
Put your final answer in \\boxed{{}}.

Problem: {problem}

Solution:"""


def execute_code(text: str) -> str:
    """Execute Python code blocks and return output."""
    blocks = re.findall(r"```python(.*?)```", text, re.DOTALL)
    if not blocks:
        return ""
    
    code = blocks[-1]
    for forbidden in ["subprocess", "os.system", "__import__"]:
        if forbidden in code:
            return f"{forbidden} not allowed"
    
    success, output = executor(code)
    return output[:500] if len(output) > 500 else output

In [ ]:
def solve(problem: str) -> int:
    """Solve problem using SC-TIR."""
    
    sampling = SamplingParams(
        temperature=config.temperature,
        max_tokens=config.max_new_tokens,
        stop=["```output\n"],
        include_stop_str_in_output=True,
    )
    
    prompt = make_prompt(problem)
    candidates = [prompt] * config.num_samples
    
    for step in range(config.num_generations):
        # Generate
        outputs = llm.generate(candidates, sampling)
        candidates = [o.prompt + o.outputs[0].text for o in outputs]
        
        # Execute code and append results
        new_candidates = []
        for text in candidates:
            if text.endswith("```output\n"):
                code_output = execute_code(text)
                text += f"{code_output}\n```\n"
            new_candidates.append(text)
        candidates = new_candidates
    
    # Extract answers
    answers = []
    for text in candidates:
        boxed = extract_boxed(text)
        if boxed:
            val = to_int(normalize(boxed))
            if val >= 0:
                answers.append(val)
    
    # Majority vote
    if not answers:
        return 0
    return Counter(answers).most_common(1)[0][0]

In [ ]:
# Main submission loop
if IS_SUBMISSION:
    # Kaggle submission
    import kaggle_evaluation.aimo_3_inference_server
    
    def predict(df):
        problem = df["problem"].iloc[0]
        answer = solve(problem)
        return pd.DataFrame({"id": df["id"], "answer": [answer]})
    
    server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
    server.serve()
else:
    # Local testing
    test_df = pd.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")
    
    correct = 0
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        pred = solve(row["problem"])
        is_correct = pred == row["answer"]
        correct += is_correct
        print(f"#{idx+1}: True={row['answer']}, Pred={pred}, {'OK' if is_correct else 'WRONG'}")
    
    print(f"\nAccuracy: {correct}/{len(test_df)} = {correct/len(test_df):.1%}")